# LeaseLens — Phase 3: PostgreSQL Verification

**Purpose:** Verify the PostgreSQL storage and retrieval layer for the Phase 3 Statute Knowledge Base.

This notebook:
- installs and starts PostgreSQL in Google Colab,
- creates the `leaselens` database and `statute_entries` table,
- loads the canonical Phase 3 statute corpus,
- verifies expected jurisdiction counts and required fields,
- tests jurisdiction-safe topic retrieval, and
- verifies evidence traceability from the database row.

**Expected corpus:** 28 entries — Maharashtra (12), Delhi (9), Central (7).

> **Security:** No password is stored in this notebook. A temporary password is generated at runtime and kept only in memory.

## 1. Install and start PostgreSQL

In [ ]:
!apt-get update -qq
!apt-get install -y postgresql postgresql-contrib -qq
!service postgresql start
!pip install -q "psycopg[binary]"
print("✅ PostgreSQL and psycopg are ready.")

## 2. Configure a temporary PostgreSQL connection

A random password is generated for this Colab runtime. It is **not printed, saved, or hard-coded** in the notebook.

In [ ]:
import secrets
import subprocess
import psycopg

DB_NAME = "leaselens"
DB_USER = "postgres"
DB_HOST = "localhost"
DB_PORT = 5432

# Runtime-only password; never print or persist it.
DB_PASSWORD = secrets.token_urlsafe(24)

sql = f"ALTER USER {DB_USER} PASSWORD {DB_PASSWORD!r};"
subprocess.run(
    ["sudo", "-u", "postgres", "psql", "-v", "ON_ERROR_STOP=1", "-c", sql],
    check=True,
    stdout=subprocess.DEVNULL,
)

exists = subprocess.run(
    ["sudo", "-u", "postgres", "psql", "-tAc",
     f"SELECT 1 FROM pg_database WHERE datname = '{DB_NAME}'"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

if exists != "1":
    subprocess.run(
        ["sudo", "-u", "postgres", "createdb", DB_NAME],
        check=True,
        stdout=subprocess.DEVNULL,
    )

def get_connection():
    return psycopg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
    )

with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT current_database(), current_user;")
        print("Connected:", cur.fetchone())

print("✅ PostgreSQL connection successful.")

## 3. Create the Phase 3 statute table

The table follows the Phase 3 schema:

`id`, `citation`, `excerpt_text`, `source_url`, `jurisdiction`, `topic_tags[]`, `last_verified_date`.

In [ ]:
CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS statute_entries (
    id TEXT PRIMARY KEY,
    citation TEXT NOT NULL,
    excerpt_text TEXT NOT NULL,
    source_url TEXT NOT NULL,
    jurisdiction TEXT NOT NULL
        CHECK (jurisdiction IN ('Maharashtra', 'Delhi', 'Central')),
    topic_tags TEXT[] NOT NULL
        CHECK (cardinality(topic_tags) > 0),
    last_verified_date DATE NOT NULL,
    CONSTRAINT statute_entries_citation_unique UNIQUE (citation)
);

CREATE INDEX IF NOT EXISTS idx_statute_entries_jurisdiction
    ON statute_entries (jurisdiction);

CREATE INDEX IF NOT EXISTS idx_statute_entries_topic_tags
    ON statute_entries USING GIN (topic_tags);
"""

with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(CREATE_TABLE_SQL)

print("✅ statute_entries table is ready.")

## 4. Upload and locate the canonical Phase 3 knowledge base

Upload the **final Phase 3 ZIP**. The notebook searches for:

`kb/leaselens_statute_kb.json`

It does not depend on the older Step-12 filename.

In [ ]:
from google.colab import files
import json
from pathlib import Path
import zipfile

uploaded = files.upload()

zip_candidates = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_candidates:
    raise FileNotFoundError("Please upload the final Phase 3 ZIP package.")

zip_path = Path("/content") / zip_candidates[0]
extract_dir = Path("/content/phase3_package")
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

kb_matches = list(extract_dir.rglob("leaselens_statute_kb.json"))
if not kb_matches:
    raise FileNotFoundError("Canonical kb/leaselens_statute_kb.json was not found in the ZIP.")

KB_PATH = kb_matches[0]
print("Using KB:", KB_PATH)

## 5. Load and validate the corpus

In [ ]:
with open(KB_PATH, "r", encoding="utf-8") as f:
    kb_data = json.load(f)

entries = kb_data["entries"] if isinstance(kb_data, dict) else kb_data

REQUIRED_FIELDS = {
    "id", "citation", "excerpt_text", "source_url",
    "jurisdiction", "topic_tags", "last_verified_date"
}
REQUIRED_TOPICS = {
    "security_deposit", "notice", "eviction",
    "maintenance", "rent_escalation", "registration"
}

assert len(entries) == 28, f"Expected 28 entries, found {len(entries)}"

for entry in entries:
    missing = REQUIRED_FIELDS - entry.keys()
    assert not missing, f"{entry.get('id')}: missing fields {missing}"
    assert entry["jurisdiction"] in {"Maharashtra", "Delhi", "Central"}
    assert entry["topic_tags"]
    assert entry["source_url"].startswith(("http://", "https://"))

topics_present = {tag for entry in entries for tag in entry["topic_tags"]}
missing_topics = REQUIRED_TOPICS - topics_present
assert not missing_topics, f"Missing required topics: {missing_topics}"

from collections import Counter
counts = Counter(entry["jurisdiction"] for entry in entries)

print("Total entries:", len(entries))
print("Maharashtra:", counts["Maharashtra"])
print("Delhi:", counts["Delhi"])
print("Central:", counts["Central"])
print("Required topics:", sorted(REQUIRED_TOPICS))
print("✅ Corpus validation passed.")

## 6. Load the corpus into PostgreSQL

In [ ]:
INSERT_SQL = """
INSERT INTO statute_entries (
    id, citation, excerpt_text, source_url,
    jurisdiction, topic_tags, last_verified_date
)
VALUES (
    %(id)s, %(citation)s, %(excerpt_text)s, %(source_url)s,
    %(jurisdiction)s, %(topic_tags)s, %(last_verified_date)s
)
ON CONFLICT (id) DO UPDATE SET
    citation = EXCLUDED.citation,
    excerpt_text = EXCLUDED.excerpt_text,
    source_url = EXCLUDED.source_url,
    jurisdiction = EXCLUDED.jurisdiction,
    topic_tags = EXCLUDED.topic_tags,
    last_verified_date = EXCLUDED.last_verified_date;
"""

with get_connection() as conn:
    with conn.cursor() as cur:
        for entry in entries:
            cur.execute(INSERT_SQL, entry)

print(f"✅ Loaded {len(entries)} Phase 3 entries into PostgreSQL.")

## 7. Verify database counts

In [ ]:
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM statute_entries;")
        total = cur.fetchone()[0]

        cur.execute("""
            SELECT jurisdiction, COUNT(*)
            FROM statute_entries
            GROUP BY jurisdiction
            ORDER BY jurisdiction;
        """)
        jurisdiction_counts = dict(cur.fetchall())

assert total == 28
assert jurisdiction_counts == {"Central": 7, "Delhi": 9, "Maharashtra": 12}

print("Total rows:", total)
for jurisdiction, count in jurisdiction_counts.items():
    print(f"{jurisdiction}: {count}")

print("✅ Database counts match the canonical corpus.")

## 8. Test jurisdiction-safe topic retrieval

Retrieval is constrained by **jurisdiction first** and **topic tag second**. A query for one pilot jurisdiction must not return entries from another pilot jurisdiction.

In [ ]:
def retrieve_statutes(jurisdiction, topic):
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT
                    id, citation, excerpt_text, source_url,
                    jurisdiction, topic_tags, last_verified_date
                FROM statute_entries
                WHERE jurisdiction = %s
                  AND %s = ANY(topic_tags)
                ORDER BY id;
            """, (jurisdiction, topic))
            return cur.fetchall()

mh_results = retrieve_statutes("Maharashtra", "security_deposit")
dl_results = retrieve_statutes("Delhi", "security_deposit")

assert mh_results, "No Maharashtra security-deposit entry retrieved."
assert dl_results, "No Delhi security-deposit entry retrieved."
assert all(row[4] == "Maharashtra" for row in mh_results)
assert all(row[4] == "Delhi" for row in dl_results)

print("Maharashtra/security_deposit:", [row[0] for row in mh_results])
print("Delhi/security_deposit:", [row[0] for row in dl_results])
print("✅ Jurisdiction isolation test passed.")

## 9. Evidence traceability test

The retrieved database row must contain the complete evidence needed downstream:

- exact citation,
- statute excerpt,
- source URL,
- jurisdiction,
- topic tags, and
- last verified date.

In [ ]:
row = mh_results[0]

entry_id, citation, excerpt, source_url, jurisdiction, topics, verified_date = row

assert entry_id
assert citation
assert excerpt
assert source_url.startswith(("http://", "https://"))
assert jurisdiction == "Maharashtra"
assert "security_deposit" in topics
assert verified_date is not None

print("Retrieved entry:", entry_id)
print("Citation:", citation)
print("Jurisdiction:", jurisdiction)
print("Topic tags:", topics)
print("Last verified:", verified_date)
print("Source URL:", source_url)
print("\n✅ PostgreSQL evidence traceability test PASSED.")

# Final verification result

## ✅ Phase 3 PostgreSQL verification: PASS

Expected final state:

- 28 statute entries loaded
- Maharashtra: 12
- Delhi: 9
- Central: 7
- All six required topic categories represented
- Jurisdiction-safe retrieval verified
- Evidence traceability verified
- `last_verified_date` preserved

> **Deployment note:** PostgreSQL runs inside the temporary Google Colab runtime. This notebook verifies the Phase 3 PostgreSQL implementation; it does not represent a permanent hosted production database.